# 🤖 Governance Bot GCP — Playbook de Despliegue y Ejemplos

**Repositorios:**
- [`GmartinezC/governance-bot-gcp`](https://github.com/GmartinezC/governance-bot-gcp) — El agente autónomo
- [`GmartinezC/governance-bot-dev-env`](https://github.com/GmartinezC/governance-bot-dev-env) — Infraestructura de pruebas + datos sintéticos

**Este notebook guía:**
1. Despliegue del agente en GCP (Cloud Run Job + Terraform)
2. Creación del ambiente de pruebas con datos sintéticos
3. Ejecución del agente y análisis de resultados
4. Ejemplos de reportes y dashboards de gobierno de datos

---
> ⚠️ **Prerequisitos:** `gcloud` CLI autenticado, `terraform >= 1.5`, `docker`, Python 3.12+


---
## 📦 Parte 1 — Despliegue del Agente (`governance-bot-gcp`)

### Arquitectura

```
[Cloud Scheduler]  ──►  [Cloud Run Job: Agentic Harness]
                              │
                    ┌─────────┼─────────┐
                    ▼         ▼         ▼
              [Planner]  [Executor]  [Tools × 8]
           Gemini 1.5 Pro  ReAct    IAM·DLP·KMS...
                    │
              [State: Firestore]  [Memory: Vertex Vector Search]
                    │
              [Output: GCS + BigQuery + Pub/Sub]
```

### Paso 1.1 — Clonar el repositorio


In [ ]:
import subprocess, os

# Definir variables globales del proyecto
PROJECT_ID = "MI_PROYECTO_GCP"   # ← cambiar por tu project ID
REGION     = "us-central1"
AGENT_DIR  = os.path.expanduser("~/Documents/proyecto-harness")
DEV_DIR    = os.path.expanduser("~/Documents/governance-bot-dev-env")

print(f"Proyecto: {PROJECT_ID}")
print(f"Región:   {REGION}")
print(f"Agent:    {AGENT_DIR}")
print(f"Dev env:  {DEV_DIR}")


In [ ]:
# Clonar el repo del agente (si no existe ya)
import os

if not os.path.exists(AGENT_DIR):
    result = subprocess.run(
        ["git", "clone", "https://github.com/GmartinezC/governance-bot-gcp.git", AGENT_DIR],
        capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
else:
    result = subprocess.run(["git", "-C", AGENT_DIR, "pull"], capture_output=True, text=True)
    print("Repo ya existe. Pull:", result.stdout.strip())


### Paso 1.2 — Configurar variables de entorno


In [ ]:
env_content = f"""GCP_PROJECT_ID={PROJECT_ID}
GCP_REGION={REGION}
FIRESTORE_DATABASE=(default)
BQ_DATASET=governance_bot
GCS_REPORTS_BUCKET={PROJECT_ID}-governance-reports
PUBSUB_TOPIC_ALERTS=governance-alerts
VERTEX_MODEL=gemini-1.5-pro
LOG_LEVEL=INFO
TARGET_PROJECTS={PROJECT_ID}
AUDIT_SCOPE=iam,dlp,catalog,lineage,policy,bq,scc,kms
"""

env_path = os.path.join(AGENT_DIR, ".env")
with open(env_path, "w") as f:
    f.write(env_content)

print(f"✅ .env escrito en {env_path}")
print()
print(env_content)


### Paso 1.3 — Desplegar infraestructura con Terraform


In [ ]:
infra_dir = os.path.join(AGENT_DIR, "infra")

# Crear bucket de state si no existe
subprocess.run(
    f"gsutil ls -b gs://{PROJECT_ID}-tfstate 2>/dev/null || "
    f"gsutil mb -p {PROJECT_ID} -l {REGION} gs://{PROJECT_ID}-tfstate",
    shell=True
)

# terraform init
result = subprocess.run(
    ["terraform", "init",
     f"-backend-config=bucket={PROJECT_ID}-tfstate",
     "-backend-config=prefix=terraform/governance-bot",
     "-input=false"],
    cwd=infra_dir, capture_output=True, text=True
)
print("INIT:", "✅ OK" if result.returncode == 0 else result.stderr[-500:])


In [ ]:
# terraform plan
result = subprocess.run(
    ["terraform", "plan",
     f"-var=project_id={PROJECT_ID}",
     f"-var=region={REGION}",
     "-var=environment=prod",
     f"-var=target_projects=[\"{PROJECT_ID}\"]",
     "-input=false"],
    cwd=infra_dir, capture_output=True, text=True
)
# Mostrar resumen del plan
lines = result.stdout.split("\n")
for line in lines:
    if any(k in line for k in ["Plan:", "will be created", "will be destroyed", "Error"]):
        print(line)


In [ ]:
# terraform apply (descomenta para ejecutar)
# result = subprocess.run(
#     ["terraform", "apply",
#      f"-var=project_id={PROJECT_ID}",
#      f"-var=region={REGION}",
#      "-var=environment=prod",
#      f"-var=target_projects=[\"{PROJECT_ID}\"]",
#      "-auto-approve", "-input=false"],
#     cwd=infra_dir, capture_output=True, text=True
# )
# print("APPLY:", "✅ OK" if result.returncode == 0 else result.stderr[-1000:])
print("⚠️  Descomenta el bloque anterior para aplicar la infraestructura.")


### Paso 1.4 — Build y push de la imagen Docker


In [ ]:
REGISTRY = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/governance-bot/governance-bot"
IMAGE_TAG = f"{REGISTRY}:latest"

print(f"Registry: {REGISTRY}")
print(f"Image:    {IMAGE_TAG}")
print()

# Comandos a ejecutar (mostrar, no ejecutar automáticamente)
cmds = [
    f"gcloud auth configure-docker {REGION}-docker.pkg.dev --quiet",
    f"docker build -t {IMAGE_TAG} {AGENT_DIR}",
    f"docker push {IMAGE_TAG}",
]
for cmd in cmds:
    print(f"$ {cmd}")


In [ ]:
# Ejecutar build (descomenta cuando estés listo)
# for cmd in cmds:
#     print(f"\n$ {cmd}")
#     result = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=AGENT_DIR)
#     if result.returncode != 0:
#         print("ERROR:", result.stderr[-500:])
#         break
#     print(result.stdout[-300:] or "OK")
print("⚠️  Descomenta el bloque anterior para hacer build y push.")


### Paso 1.5 — Primer run manual del agente


In [ ]:
run_cmd = (
    f"gcloud run jobs execute governance-bot "
    f"--region {REGION} "
    f"--project {PROJECT_ID} "
    f"--wait"
)
print(f"Comando para ejecutar el agente:")
print(f"$ {run_cmd}")
print()
print("Ver logs en tiempo real:")
print(f"$ gcloud logging read 'resource.type=cloud_run_job AND resource.labels.job_name=governance-bot' "
      f"--project={PROJECT_ID} --limit=50 --format='value(textPayload)'")


---
## 🌱 Parte 2 — Ambiente de Pruebas (`governance-bot-dev-env`)

Crea recursos GCP con **problemas de gobierno intencionales** para validar
que el agente detecta los findings correctamente.

### Recursos sintéticos y findings esperados

| Recurso | Problema intencional | Finding esperado |
|---------|---------------------|-----------------|
| SA `test-overprivileged-sa` | `roles/editor` (primitive role) | **IAM HIGH** |
| Tabla `users_pii` | RUT, emails, tarjetas de crédito | **DLP HIGH** |
| Tabla `payments` | Números de tarjeta | **DLP HIGH** |
| Buckets `test-landing`, `test-exports` | Sin CMEK | **KMS MEDIUM** |
| Datasets `test_data_raw`, `customer_data` | Sin labels ni descripción | **BQ LOW** |

### Paso 2.1 — Clonar y desplegar


In [ ]:
if not os.path.exists(DEV_DIR):
    result = subprocess.run(
        ["git", "clone", "https://github.com/GmartinezC/governance-bot-dev-env.git", DEV_DIR],
        capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
else:
    result = subprocess.run(["git", "-C", DEV_DIR, "pull"], capture_output=True, text=True)
    print("Repo ya existe. Pull:", result.stdout.strip())


In [ ]:
dev_infra = os.path.join(DEV_DIR, "terraform")

# terraform init
result = subprocess.run(
    ["terraform", "init",
     f"-backend-config=bucket={PROJECT_ID}-tfstate",
     "-backend-config=prefix=terraform/governance-bot-dev",
     "-input=False"],
    cwd=dev_infra, capture_output=True, text=True
)
print("INIT:", "✅ OK" if result.returncode == 0 else result.stderr[-500:])


In [ ]:
# terraform apply del ambiente de pruebas (descomenta para ejecutar)
# result = subprocess.run(
#     ["terraform", "apply",
#      f"-var=project_id={PROJECT_ID}",
#      f"-var=region={REGION}",
#      "-var=environment=dev",
#      "-auto-approve", "-input=false"],
#     cwd=dev_infra, capture_output=True, text=True
# )
# print("APPLY:", "✅ OK" if result.returncode == 0 else result.stderr[-1000:])
print("⚠️  Descomenta para crear los recursos de prueba con Terraform.")


### Paso 2.2 — Cargar datos sintéticos


In [ ]:
seed_script = os.path.join(DEV_DIR, "scripts", "seed_data.py")

# Ver qué genera el script
print("Ejecutar seed de datos sintéticos:")
print(f"$ python3 {seed_script} --project {PROJECT_ID} --rows 500")
print()
print("Datos que se generan:")
data_summary = {
    "test_data_raw.transactions": "500 transacciones (sin PII directo)",
    "test_data_raw.users_pii":    "200 usuarios con RUT, email, teléfono, tarjeta",
    "customer_data.orders":       "300 órdenes de compra",
    "customer_data.payments":     "300 pagos con card_number",
    "GCS landing/exports":        "4 archivos CSV con datos sensibles",
    "Firestore":                  "Estado inicial del proyecto (nunca auditado)",
}
for table, desc in data_summary.items():
    print(f"  📊 {table:40s} {desc}")


In [ ]:
# Ejecutar seed (descomenta cuando la infra esté lista)
# result = subprocess.run(
#     ["python3", seed_script, "--project", PROJECT_ID, "--rows", "500"],
#     capture_output=True, text=True
# )
# print(result.stdout or result.stderr)
print("⚠️  Descomenta para cargar los datos sintéticos.")


---
## ▶️ Parte 3 — Ejecutar el Agente y Ver Resultados

### Paso 3.1 — Ejecutar localmente (para desarrollo)


In [ ]:
# Instalar dependencias del agente
result = subprocess.run(
    ["pip", "install", "-q", "-r", "requirements.txt"],
    cwd=AGENT_DIR, capture_output=True, text=True
)
print("pip install:", "✅ OK" if result.returncode == 0 else result.stderr[-300:])


In [ ]:
# Ejecutar agente localmente (requiere gcloud auth application-default login)
run_local = [
    "bash", "-c",
    f"cd {AGENT_DIR} && python -m agent.main"
]
print("Para ejecutar localmente:")
print(f"$ cd {AGENT_DIR}")
print("$ gcloud auth application-default login")
print("$ python -m agent.main")
print()
print("Para Cloud Run:")
print(f"$ gcloud run jobs execute governance-bot --region {REGION} --project {PROJECT_ID} --wait")


### Paso 3.2 — Monitorear en Cloud Logging


In [ ]:
log_filter = (
    f"resource.type=cloud_run_job "
    f"AND resource.labels.job_name=governance-bot"
)
log_cmd = (
    f"gcloud logging read '{log_filter}' "
    f"--project={PROJECT_ID} "
    f"--limit=100 "
    f"--format='json'"
)
print("Comando para ver logs:")
print(f"$ {log_cmd}")
print()

# Ejemplo de log estructurado que genera el agente:
sample_log = {
    "timestamp": "2026-05-08T14:30:00Z",
    "severity": "INFO",
    "event": "tool_completed",
    "tool": "iam_analyzer",
    "project_id": "mi-proyecto",
    "findings": 3,
    "score": 60.0
}
import json
print("Ejemplo de log estructurado (JSON):")
print(json.dumps(sample_log, indent=2))


---
## 📄 Parte 4 — Ejemplo de Reporte de Gobierno de Datos

El agente genera reportes Markdown que se guardan en GCS. Aquí un ejemplo
del output esperado para el ambiente de pruebas.


In [ ]:
from IPython.display import Markdown, display
from datetime import datetime

sample_report = """
# Reporte de Gobierno de Datos — {project_id}
**Fecha:** {date}  |  **Run ID:** `{run_id}`

## Resumen de Scores

| Dominio | Score | Estado |
|---------|-------|--------|
| IAM     | 40.0  | 🔴 Crítico |
| DLP     | 20.0  | 🔴 Crítico |
| Catalog | 55.0  | 🟡 Atención |
| Lineage | 30.0  | 🔴 Crítico |
| Policy  | 60.0  | 🟡 Atención |
| BQ      | 45.0  | 🔴 Crítico |
| SCC     | 70.0  | 🟡 Atención |
| KMS     | 35.0  | 🔴 Crítico |

**Score General: 43.1 🔴**

---

## Hallazgos Críticos

### 🔴 HIGH — Primitive role 'roles/editor' asignado
- **Recurso:** `serviceAccount:test-overprivileged-sa@{project_id}.iam.gserviceaccount.com`
- **Dominio:** IAM
- **Descripción:** El role primitivo 'roles/editor' otorga acceso excesivo a todos los recursos del proyecto.
- **Recomendación:** Reemplazar con roles granulares como `roles/bigquery.dataViewer`, `roles/storage.objectViewer`.

### 🔴 HIGH — Datos sensibles detectados: CREDIT_CARD_NUMBER
- **Recurso:** `{project_id}.customer_data.payments`
- **Dominio:** DLP
- **Descripción:** Se encontraron 187 instancias de CREDIT_CARD_NUMBER en la tabla `payments`.
- **Recomendación:** Aplicar tokenización o enmascaramiento con Cloud DLP. Revisar si la columna `card_number` debe existir.

### 🔴 HIGH — Datos sensibles detectados: CHILE_RUT
- **Recurso:** `{project_id}.test_data_raw.users_pii`
- **Dominio:** DLP
- **Descripción:** Se encontraron 200 instancias de CHILE_RUT y 200 de EMAIL_ADDRESS.
- **Recomendación:** Clasificar datos con tags en Data Catalog. Aplicar column-level security en BigQuery.

### 🟡 MEDIUM — Dataset BigQuery sin CMEK: test_data_raw
- **Recurso:** `{project_id}.test_data_raw`
- **Dominio:** KMS
- **Descripción:** El dataset usa cifrado por defecto de Google, no CMEK.
- **Recomendación:** Configurar `default_encryption_configuration` con una Cloud KMS key.

## Top 5 Recomendaciones

1. Remover `roles/editor` del SA `test-overprivileged-sa` y asignar roles granulares.
2. Aplicar enmascaramiento DLP a columnas `card_number`, `rut`, `email` en tablas de producción.
3. Configurar CMEK en datasets BigQuery `test_data_raw` y `customer_data`.
4. Agregar labels de clasificación a todos los datasets BigQuery.
5. Habilitar `uniformBucketLevelAccess` en buckets `test-landing` y `test-exports`.
""".format(
    project_id=PROJECT_ID,
    date=datetime.now().strftime("%Y-%m-%d %H:%M UTC"),
    run_id="550e8400-e29b-41d4-a716-446655440000"
)

display(Markdown(sample_report))


---
## 📊 Parte 5 — Dashboards de Gobierno de Datos

Visualizaciones del estado de gobierno basadas en los resultados del agente.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Scores del run de ejemplo (ambiente de pruebas con problemas intencionales)
domains = ["IAM", "DLP", "Catalog", "Lineage", "Policy", "BQ", "SCC", "KMS"]
scores  = [40.0, 20.0, 55.0, 30.0, 60.0, 45.0, 70.0, 35.0]

def score_color(s):
    if s > 80: return "#2E7D32"
    if s > 50: return "#F57F17"
    return "#C62828"

colors = [score_color(s) for s in scores]
overall = np.mean(scores)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor("#F8F9FA")

# ── Gráfico 1: Barras por dominio ──
ax1 = axes[0]
bars = ax1.barh(domains, scores, color=colors, height=0.6, edgecolor="white", linewidth=1.5)
ax1.set_xlim(0, 100)
ax1.set_xlabel("Score (0–100)", fontsize=11)
ax1.set_title(f"Governance Score por Dominio\nScore General: {overall:.1f}/100",
              fontsize=13, fontweight="bold", pad=15)
ax1.axvline(50, color="#9E9E9E", linestyle="--", alpha=0.5, linewidth=1)
ax1.axvline(80, color="#9E9E9E", linestyle="--", alpha=0.5, linewidth=1)
ax1.text(51, -0.8, "50", color="#9E9E9E", fontsize=8)
ax1.text(81, -0.8, "80", color="#9E9E9E", fontsize=8)
ax1.set_facecolor("#FAFAFA")

for bar, score in zip(bars, scores):
    ax1.text(score + 1.5, bar.get_y() + bar.get_height()/2,
             f"{score:.0f}", va="center", fontsize=10, fontweight="bold",
             color=score_color(score))

legend = [
    mpatches.Patch(color="#2E7D32", label="Óptimo (>80)"),
    mpatches.Patch(color="#F57F17", label="Atención (50–80)"),
    mpatches.Patch(color="#C62828", label="Crítico (<50)"),
]
ax1.legend(handles=legend, loc="lower right", fontsize=9)

# ── Gráfico 2: Radar chart ──
ax2 = axes[1]
ax2.set_facecolor("#FAFAFA")
angles = np.linspace(0, 2*np.pi, len(domains), endpoint=False).tolist()
values = scores + [scores[0]]
angles += angles[:1]

ax2 = plt.subplot(122, polar=True)
ax2.set_facecolor("#FAFAFA")
ax2.plot(angles, values, "o-", linewidth=2, color="#1565C0")
ax2.fill(angles, values, alpha=0.25, color="#42A5F5")
ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(domains, fontsize=10)
ax2.set_ylim(0, 100)
ax2.set_yticks([20, 40, 60, 80])
ax2.set_yticklabels(["20", "40", "60", "80"], fontsize=7, color="#757575")
ax2.set_title("Radar de Gobierno\n(ambiente de pruebas)",
              fontsize=12, fontweight="bold", pad=20)

# Zona óptima
angles_full = np.linspace(0, 2*np.pi, 100)
ax2.fill(angles_full, [80]*100, alpha=0.05, color="#2E7D32")

plt.tight_layout(pad=3)
plt.savefig("/tmp/governance_scores.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\n📊 Score general: {overall:.1f}/100 — Ambiente de pruebas (con problemas intencionales)")


### Dashboard 2 — Findings por severidad y dominio


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor("#F8F9FA")

# ── Findings por severidad ──
ax = axes[0]
severities = ["CRITICAL", "HIGH", "MEDIUM", "LOW"]
counts     = [0, 5, 8, 12]
sev_colors = ["#B71C1C", "#C62828", "#F57F17", "#1565C0"]
wedges, texts, autotexts = ax.pie(
    counts, labels=severities, colors=sev_colors,
    autopct="%1.0f%%", startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 2}
)
for at in autotexts:
    at.set_fontsize(10); at.set_fontweight("bold"); at.set_color("white")
ax.set_title(f"Findings por Severidad\nTotal: {sum(counts)}", fontsize=12, fontweight="bold")
ax.set_facecolor("#FAFAFA")

# ── Findings por dominio ──
ax = axes[1]
dom_findings = {
    "IAM": 3, "DLP": 5, "KMS": 4,
    "BQ": 6, "Policy": 2, "Catalog": 3, "Lineage": 1, "SCC": 1
}
dom_colors_list = [score_color(scores[domains.index(d)]) for d in dom_findings.keys()]
bars = ax.bar(dom_findings.keys(), dom_findings.values(),
              color=dom_colors_list, edgecolor="white", linewidth=1.5)
ax.set_title("Findings por Dominio", fontsize=12, fontweight="bold")
ax.set_ylabel("Cantidad de findings")
ax.set_facecolor("#FAFAFA")
for bar, val in zip(bars, dom_findings.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(val), ha="center", fontsize=10, fontweight="bold")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

# ── Tendencia histórica ──
ax = axes[2]
runs = ["Run 1\n(baseline)", "Run 2\n(+1 sem)", "Run 3\n(+2 sem)", "Run 4\n(actual)"]
trend_iam = [40, 45, 55, 65]
trend_dlp = [20, 20, 35, 50]
trend_kms = [35, 35, 50, 60]
trend_gen = [43, 48, 56, 64]
ax.plot(runs, trend_iam, "o-", label="IAM",     color="#C62828", linewidth=2)
ax.plot(runs, trend_dlp, "s-", label="DLP",     color="#E65100", linewidth=2)
ax.plot(runs, trend_kms, "^-", label="KMS",     color="#F57F17", linewidth=2)
ax.plot(runs, trend_gen, "D-", label="General", color="#1565C0", linewidth=2.5, linestyle="--")
ax.axhline(80, color="#2E7D32", linestyle=":", alpha=0.5, linewidth=1.5, label="Target (80)")
ax.fill_between(range(len(runs)), 80, 100, alpha=0.05, color="#2E7D32")
ax.set_ylim(0, 100)
ax.set_title("Tendencia de Scores\n(4 semanas)", fontsize=12, fontweight="bold")
ax.set_ylabel("Score (0–100)")
ax.legend(loc="lower right", fontsize=8)
ax.set_facecolor("#FAFAFA")
ax.grid(True, alpha=0.3)

plt.suptitle(f"Dashboard de Gobierno de Datos — {PROJECT_ID}",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("/tmp/governance_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()


### Dashboard 3 — Tabla resumen de recursos auditados


In [ ]:
import pandas as pd

# Tabla resumen de findings (simulada)
findings_data = {
    "Severidad": ["HIGH", "HIGH", "HIGH", "MEDIUM", "MEDIUM", "MEDIUM", "LOW", "LOW", "LOW"],
    "Dominio":   ["IAM",  "DLP",  "DLP",  "KMS",    "KMS",    "Policy", "BQ",  "BQ",  "Catalog"],
    "Recurso": [
        "test-overprivileged-sa",
        "customer_data.payments",
        "test_data_raw.users_pii",
        "PROJECT-test-landing",
        "PROJECT-test-exports",
        "PROJECT-test-landing",
        "test_data_raw",
        "customer_data",
        "users_pii (sin tags)",
    ],
    "Finding": [
        "Primitive role roles/editor",
        "187 números de tarjeta (CREDIT_CARD_NUMBER)",
        "200 RUT + 200 emails (CHILE_RUT, EMAIL)",
        "Bucket sin CMEK",
        "Bucket sin CMEK",
        "Sin uniformBucketLevelAccess",
        "Dataset sin labels ni descripción",
        "Dataset sin labels ni descripción",
        "Tabla sin tags Data Catalog",
    ],
    "Remediado": ["❌", "❌", "❌", "❌", "❌", "❌", "❌", "❌", "❌"]
}

df = pd.DataFrame(findings_data)
df["Recurso"] = df["Recurso"].str.replace("PROJECT", PROJECT_ID[:10]+"...")

# Colorear por severidad
def highlight_severity(row):
    colors = {"HIGH": "background-color: #FFEBEE", "MEDIUM": "background-color: #FFF3E0", "LOW": "background-color: #E3F2FD"}
    return [colors.get(row["Severidad"], "")] * len(row)

display(df.style.apply(highlight_severity, axis=1).set_caption("📋 Findings detectados — ambiente de pruebas"))


---
## 🚀 Próximos Pasos

1. **Conectar credenciales GCP:** `gcloud auth application-default login`
2. **Crear tfstate bucket y aplicar Terraform** (Partes 1.3 y 2.1)
3. **Build y push de la imagen Docker** (Parte 1.4)
4. **Cargar datos sintéticos** (Parte 2.2)
5. **Ejecutar el agente** y revisar los findings reales en BigQuery
6. **Conectar Looker Studio** al dataset `governance_bot` para dashboard en producción

### Conectar Looker Studio al BigQuery

```
1. Ir a https://lookerstudio.google.com
2. Crear nuevo reporte → Fuente de datos → BigQuery
3. Seleccionar: [PROJECT_ID] → governance_bot → governance_runs
4. Crear gráficas de:
   - Score general por fecha (línea temporal)
   - Scores por dominio (barras)
   - Findings por severidad (pie chart)
   - Top proyectos con peores scores (tabla)
```

---
*Notebook generado automáticamente para el proyecto governance-bot-gcp*
